# Historical Max-c Diagnostics over D and T

> **Archived analysis:** these values are not estimates of the experimental contraction rate and must not be used as a heatmap success criterion. New heatmap runs use only a preset fixed `c`.

This notebook reads legacy `c_bound` files to support historical and exploratory analysis only.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
# Match a legacy generated result.
n = 100
q = 0.8
beta = 0.005
D_min = 1
D_max = 30
c_target = 1e-3
num_samples = 10
T_intervals = 1000
T_max = 20000
corruption_type = "sup_c"
c_success_mode = "bound"
bound_rule = "max_c_no_failure"

In [ ]:
working_dir = Path.cwd().resolve()
search_roots = (working_dir, *working_dir.parents)
repo_candidates = (*search_roots, *(path / "adv_D_upper" for path in search_roots))
repo_root = next(
    (path for path in repo_candidates if (path / "heat_map_raw_data").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f"Could not find heat_map_raw_data from {working_dir} or its parents. "
        "Start Jupyter inside the adv_D_upper project."
    )
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
data_dir = repo_root / "heat_map_raw_data"
beta_for_filename = np.floor(beta * 100) / 100
base_suffix = (
    f"__n={n}__q={q * 100:.0f}__beta={beta_for_filename * 100:.0f}"
    f"__D_min={D_min}__D_max={D_max}__c={c_target:1.0e}"
    f"__num_samples={num_samples}__T_intervals={T_intervals}__T_max={T_max}"
    f"__corruption_type={corruption_type}"
)
mode_suffix = f"__c_success={c_success_mode}__rule={bound_rule}"
c_path = data_dir / f"D_vs_T__c_bound{base_suffix}{mode_suffix}.txt"
alpha0_path = data_dir / f"D_vs_T__c_bound_alpha0{base_suffix}{mode_suffix}.txt"
alpha_prime_path = data_dir / f"D_vs_T__c_bound_alpha_prime{base_suffix}{mode_suffix}.txt"
failure_prob_path = data_dir / f"D_vs_T__c_bound_failure_prob{base_suffix}{mode_suffix}.txt"
d_min_path = data_dir / f"D_vs_T__D_MIN{base_suffix}.txt"

print("project root:", repo_root)
print("Python import root added:", repo_root)
print("c bound path:", c_path)
if not c_path.exists():
    candidates = sorted(data_dir.glob("D_vs_T__c_bound*.txt"))
    candidate_text = "\n".join(f"  - {path.name}" for path in candidates)
    raise FileNotFoundError(
        f"Expected max-c file does not exist:\n{c_path}\n"
        f"Available c-bound files:\n{candidate_text or '  (none)'}"
    )

In [ ]:
T_values = np.arange(T_intervals, T_max + 1, T_intervals)
D_values = np.arange(D_min, D_max + 1)
expected_shape = (len(D_values), len(T_values))

def load_d_by_t(path, name):
    values = np.atleast_2d(np.loadtxt(path))
    if values.shape == expected_shape:
        return values
    if values.T.shape == expected_shape:
        print(f"Transposing {name}: loaded shape {values.shape}")
        return values.T
    raise ValueError(
        f"{name} has shape {values.shape}; expected D x T = {expected_shape}. "
        "Check the parameter cell against the generated data file."
    )

c_bound = load_d_by_t(c_path, "c_bound")
alpha_0 = load_d_by_t(alpha0_path, "alpha_0")
alpha_prime = load_d_by_t(alpha_prime_path, "alpha_prime")
failure_prob = load_d_by_t(failure_prob_path, "failure_prob")
d_min_values = np.ravel(np.loadtxt(d_min_path)) if d_min_path.exists() else None
if d_min_values is not None and len(d_min_values) != len(T_values):
    raise ValueError(
        f"D_min has length {len(d_min_values)}; expected {len(T_values)} T values."
    )

print("shape:", c_bound.shape)
print("c min / mean / max:", np.nanmin(c_bound), np.nanmean(c_bound), np.nanmax(c_bound))
print("alpha_0 min / mean / max:", np.nanmin(alpha_0), np.nanmean(alpha_0), np.nanmax(alpha_0))
print("alpha_prime min / mean / max:", np.nanmin(alpha_prime), np.nanmean(alpha_prime), np.nanmax(alpha_prime))
print("ignored failure probability min / mean / max:", np.nanmin(failure_prob), np.nanmean(failure_prob), np.nanmax(failure_prob))

In [ ]:
T_edges = np.r_[T_values - T_intervals / 2, T_values[-1] + T_intervals / 2]
D_edges = np.r_[D_values - 0.5, D_values[-1] + 0.5]

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, values, title, label in (
    (axes[0], c_bound, "Historical max c (not a success criterion)", "c"),
    (axes[1], alpha_0, "Selected alpha_0", "alpha_0"),
    (axes[2], failure_prob, "Ignored failure probability", "failure probability"),
):
    image = ax.pcolormesh(T_edges, D_edges, values, shading="flat", cmap="viridis")
    fig.colorbar(image, ax=ax, label=label)
    ax.set_title(title)
    ax.set_xlabel("T")
axes[0].set_ylabel("D")
if d_min_values is not None:
    axes[0].plot(T_values, d_min_values, color="black", linewidth=3, label="theoretical D_min")
    axes[0].legend()
fig.tight_layout()